In [ ]:
#clone Mask R-CNN github repo
!git clone https://github.com/matterport/Mask_RCNN.git

In [ ]:
#install dependencies and import functions
!pip install tensorflow
!pip install imgaug
!pip install pycocotools
!pip install numpy==1.20.3
!pip install seaborn
!pip install sklearn
!pip install scikit-learn

import os
import sys
import random
import math
import numpy as np
import skimage.io
import matplotlib
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model  # Updated import
#from tensorflow.keras import engine as KE  # If specific modules or functions are needed, adjust accordingly
import json
import datetime
import skimage.draw
import cv2
from mrcnn.visualize import display_instances
import imgaug
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')
import re
import time
import matplotlib.patches as patches
import matplotlib.image as mpimg
import skimage


In [2]:
# Root directory of the project
ROOT_DIR = os.path.abspath("/Applications/anaconda3/envs/COMP6970/FinalProject/MaskRCNN/Mask_RCNN") #update as needed

In [3]:
# Import Mask R-CNN
sys.path.append(ROOT_DIR)  # To find local version of the library
from mrcnn import utils
import mrcnn.model as modellib
from mrcnn import visualize
from mrcnn.config import Config
from mrcnn import model as modellib, utils
from mrcnn.visualize import display_images
from mrcnn.visualize import display_instances
from mrcnn.model import log

In [ ]:
# Path to trained weights file
COCO_WEIGHTS_PATH = os.path.join(ROOT_DIR, "mask_rcnn_coco.h5") #pre-trained Mask R-CNN model

# Directory to save logs and model checkpoints
DEFAULT_LOGS_DIR = os.path.join(ROOT_DIR, "logs")

In [ ]:
#TRAINING CONFIGURATION 

class CustomConfig(Config):
    """Configuration for training on the custom  dataset.
    Derives from the base Config class and overrides some values.
    """
    # Give the configuration a recognizable name
    NAME = "algae"

    # NUMBER OF GPUs to use. When using only a CPU, this needs to be set to 1.
    GPU_COUNT = 1
    
    # We use a GPU with 12GB memory, which can fit two images.
    # Adjust down if you use a smaller GPU.
    IMAGES_PER_GPU = 2
    
    # Number of classes (including background)
    NUM_CLASSES = 1 + 25  # Background + 25 algae classes

    # Number of training steps per epoch
    STEPS_PER_EPOCH = 5

    # Skip detections with < 90% confidence
    DETECTION_MIN_CONFIDENCE = 0.9
    
    LEARNING_RATE = 0.001

In [ ]:
#LOAD CUSTOM DATA FOR TRAINING

class CustomDataset(utils.Dataset):

    def load_custom(self, dataset_dir, subset):
        #dataset_dir: Root directory of the dataset.
        #subset: Subset to load: train or val
              
        # Add classes 
        self.add_class("algae", 1, "Actinoptychus")
        self.add_class("algae", 2, "Bacillaria")
        self.add_class("algae", 3, "Biddulphia")
        self.add_class("algae", 4, "Centric Diatom")
        self.add_class("algae", 5, "Ciliate")
        self.add_class("algae", 6, "Coscinodiscus")
        self.add_class("algae", 7, "Cylindrotheca")
        self.add_class("algae", 8, "Dactyliosolen")
        self.add_class("algae", 9, "Diatom")
        self.add_class("algae", 10, "Dinoflagellate")
        self.add_class("algae", 11, "Entomoneis")
        self.add_class("algae", 12, "Euglenoid")
        self.add_class("algae", 13, "Fragilidium")
        self.add_class("algae", 14, "Hemiaulus")
        self.add_class("algae", 15, "Heterosigma Akashiwo")
        self.add_class("algae", 16, "Lyrella")
        self.add_class("algae", 17, "Navicula")
        self.add_class("algae", 18, "Nitzschia")
        self.add_class("algae", 19, "Odontella")
        self.add_class("algae", 20, "Paralia")
        self.add_class("algae", 21, "Pennate Diatom")
        self.add_class("algae", 22, "Pleurosigma")
        self.add_class("algae", 23, "Prorocentrum")
        self.add_class("algae", 24, "Tintinnid")
        self.add_class("algae", 25, "Tripos Hircus")
     
        # Path to train and val folders in dataset
        assert subset in ["train", "val"]
        dataset_dir = os.path.join(dataset_dir, subset)

        # Load training data json file (annotations)
        annotations1 = json.load(open("/Applications/anaconda3/envs/COMP6970/FinalProject/MaskRCNN/Mask_RCNN/Data/train_20shot/train20.json"))
        
        # print(annotations1)
        annotations = list(annotations1.values())  # don't need the dict keys

        # Skip unannotated images.
        annotations = [a for a in annotations if a['regions']]
        
        # Add images
        for a in annotations:
            # print(a)
            # Get the x, y coordinates of points of the polygons that make up
            # the outline of each object instance. There are stores in the
            # shape_attributes (see json format above)
            polygons = [r['shape_attributes'] for r in a['regions']] 
            objects = [s['region_attributes']['names'] for s in a['regions']]
            print("algae:",objects)
            name_dict = {"Actinoptychus":1,"Bacillaria":2, "Biddulphia":3, "Centric Diatom":4, "Ciliate":5, "Coscinodiscus":6, 
                        "Cylindrotheca":7, "Dactyliosolen":8, "Diatom":9, "Dinoflagellate":10, "Entomoneis":11, "Euglenoid":12, 
                        "Fragilidium":13, "Hemiaulus":14, "Heterosigma Akashiwo":15, "Lyrella":16, "Navicula":17, "Nitzschia":18,
                        "Odontella":19, "Paralia":20, "Pennate Diatom":21, "Pleurosigma":22, "Prorocentrum":23, "Tintinnid":24, 
                        "Tripos Hircus":25}

            # key = tuple(name_dict)
            num_ids = [name_dict[a] for a in objects]
     
            # num_ids = [int(n['Event']) for n in objects]
            # load_mask() needs the image size to convert polygons to masks.
            print("numids",num_ids)
            image_path = os.path.join(dataset_dir, a['filename'])
            image = skimage.io.imread(image_path)
            height, width = image.shape[:2]

            self.add_image(
                "algae",  ## for a single class just add the name here
                image_id=a['filename'],  # use file name as a unique image id
                path=image_path,
                width=width, height=height,
                polygons=polygons,
                num_ids=num_ids
                )

    def load_mask(self, image_id):
        """Generate instance masks for an image.
       Returns:
        masks: A bool array of shape [height, width, instance count] with
            one mask per instance.
        class_ids: a 1D array of class IDs of the instance masks.
        """
        # delegate image to parent class
        image_info = self.image_info[image_id]
        if image_info["source"] != "algae":
            return super(self.__class__, self).load_mask(image_id)

        # Convert polygons to a bitmap mask of shape
        # [height, width, instance_count]
        info = self.image_info[image_id]
        if info["source"] != "algae":
            return super(self.__class__, self).load_mask(image_id)
        num_ids = info['num_ids']
        mask = np.zeros([info["height"], info["width"], len(info["polygons"])],
                        dtype=np.uint8)
        for i, p in enumerate(info["polygons"]):
            # Get indexes of pixels inside the polygon and set them to 1
        	rr, cc = skimage.draw.polygon(p['all_points_y'], p['all_points_x'])

        	mask[rr, cc, i] = 1

        # Return mask, and array of class IDs of each instance. 
        # Map class names to class IDs.
        num_ids = np.array(num_ids, dtype=np.int32)
        return mask, num_ids #np.ones([mask.shape[-1]], dtype=np.int32)

    def image_reference(self, image_id):
        """Return the path of the image."""
        info = self.image_info[image_id]
        if info["source"] == "algae":
            return info["path"]
        else:
            super(self.__class__, self).image_reference(image_id)


In [ ]:
# TRAIN MODEL

def train(model):
    """Train the model."""
    # Training dataset.
    dataset_train = CustomDataset()
    dataset_train.load_custom("/Applications/anaconda3/envs/COMP6970/FinalProject/MaskRCNN/Mask_RCNN/Data", "train_20shot")
    dataset_train.prepare()

    # Validation dataset
    dataset_val = CustomDataset()
    dataset_val.load_custom("/Applications/anaconda3/envs/COMP6970/FinalProject/MaskRCNN/Mask_RCNN/Data", "val_20shot")
    dataset_val.prepare()

    model.train(dataset_train, dataset_val,
                learning_rate=config.LEARNING_RATE,
                epochs=10, #update # epochs as needed
                layers='heads') #we are just training the networ heads instead of all the layers since this model is pre-trained
                                #layers = 'all' if training all layers
				
    
config = CustomConfig()
model = modellib.MaskRCNN(mode="training", config=config,
                                  model_dir=DEFAULT_LOGS_DIR)

weights_path = COCO_WEIGHTS_PATH
        # Download weights file
if not os.path.exists(weights_path):
  utils.download_trained_weights(weights_path)

#load pre-trained weights and fine tune on algae data
model.load_weights(weights_path, by_name=True, exclude=[
            "mrcnn_class_logits", "mrcnn_bbox_fc",
            "mrcnn_bbox", "mrcnn_mask"])

train(model)	

In [ ]:
#TEST MODEL
#test model on algae data

# Create a session with device placement logs
sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(log_device_placement=True))
print(sess)

# GPU for training.
DEVICE = "/cpu:0"  # /cpu:0 or /gpu:0


class InferenceConfig(CustomConfig):
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1
    #Minimum probability value to accept a detected instance
    # ROIs below this threshold are skipped
    DETECTION_MIN_CONFIDENCE = 0.7

    # Non-maximum suppression threshold for detection
    DETECTION_NMS_THRESHOLD = 0.3

inference_config = InferenceConfig()

# Recreate the model in inference mode
model = modellib.MaskRCNN(mode="inference", 
                          config=inference_config,
                          model_dir=DEFAULT_LOGS_DIR)

# Get path to saved weights
# Either set a specific path or find last trained weights
# model_path = os.path.join(ROOT_DIR, ".h5 file name here")


#model_path = model.find_last()
model_path = '/Applications/anaconda3/envs/COMP6970/FinalProject/MaskRCNN/Mask_RCNN/logs/mask_rcnn_object_0004.h5'


# Load trained weights
print("Loading weights from ", model_path)
model.load_weights(model_path, by_name=True)

#test on images
real_test_dir = '/Applications/anaconda3/envs/COMP6970/FinalProject/MaskRCNN/Mask_RCNN/Data/test'
image_paths = []
for filename in os.listdir(real_test_dir):
    if os.path.splitext(filename)[1].lower() in ['.png', '.jpg', '.jpeg']:
        image_paths.append(os.path.join(real_test_dir, filename))

for image_path in image_paths:
    img = skimage.io.imread(image_path)
    img_arr = np.array(img)
    results = model.detect([img_arr], verbose=1)
    r = results[0]
    visualize.display_instances(img, r['rois'], r['masks'], r['class_ids'], 
                                dataset_val.class_names, r['scores'], figsize=(5,5))

In [ ]:
#METRICS

#confusion matrix
config=inference_config
dataset = dataset_val

gt_tot = np.array([])
pred_tot = np.array([])

#mAP list
mAP_ = []

#compute gt_tot, pred_tot and mAP for each image in the test dataset
for image_id in dataset.image_ids:
    image, image_meta, gt_class_id, gt_bbox, gt_mask =\
        modellib.load_image_gt(dataset, config, image_id)#, #use_mini_mask=False)
    info = dataset.image_info[image_id]

    # Run the model
    results = model.detect([image], verbose=1)
    r = results[0]
    
    #compute gt_tot and pred_tot
    gt, pred = utils.gt_pred_lists(gt_class_id, gt_bbox, r['class_ids'], r['rois'])
    gt_tot = np.append(gt_tot, gt)
    pred_tot = np.append(pred_tot, pred)
    
    #precision_, recall_, AP_ 
    AP_, precision_, recall_, overlap_ = utils.compute_ap(gt_bbox, gt_class_id, gt_mask,
                                          r['rois'], r['class_ids'], r['scores'], r['masks'])
    #check if the vectors len are equal
    print("the actual len of the gt vect is : ", len(gt_tot))
    print("the actual len of the pred vect is : ", len(pred_tot))
    
    mAP_.append(AP_)
    print("Average precision of this image : ",AP_)
    print("The actual mean average precision for the whole images", sum(mAP_)/len(mAP_))

import pandas as pd
gt_tot=gt_tot.astype(int)
pred_tot=pred_tot.astype(int)

#save the vectors of gt and pred
save_dir = "output"
gt_pred_tot_json = {"gt_tot" : gt_tot, "pred_tot" : pred_tot}
df = pd.DataFrame(gt_pred_tot_json)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
df.to_json(os.path.join(save_dir,"gt_pred_test.json"))

tp,fp,fn=utils.plot_confusion_matrix_from_data(gt_tot,pred_tot,columns=["bg","Actinoptychus","Bacillaria","Biddulphia","Centric Diatom",
                                                                        "Ciliate","Coscinodiscus","Cylindrotheca","Dactyliosolen","Diatom", 
                                                                        "Dinoflagellate","Entomoneis","Euglenoid","Fragilidium","Hemiaulus",
                                                                        "Heterosigma Akashiwo","Lyrella","Navicula","Nitzschia","Odontella",
                                                                        "Paralia","Pennate Diatom","Pleurosigma","Prorocentrum","Tintinnid", 
                                                                        "Tripos Hircus"] ,fz=18, figsize=(20,20), lw=0.5)
#Accuracy, precision, and recall
print("tp for each class :",tp)
print("fp for each class :",fp)
print("fn for each class :",fn)

#eliminate the background class from tps fns and fns lists since it doesn't concern us anymore : 
del tp[0]
del fp[0]
del fn[0]
print("\n########################\n")
print("tp for each class :",tp)
print("fp for each class :",fp)
print("fn for each class :",fn)

accuracy, precisions, recalls, overlaps = utils.compute_ap(gt_bbox, gt_class_id, gt_mask, r['rois'], r['class_ids'], r['scores'], r['masks'])

In [ ]:
#cite Mask R-CNN
@misc{matterport_maskrcnn_2017,
  title={Mask R-CNN for object detection and instance segmentation on Keras and TensorFlow},
  author={Waleed Abdulla},
  year={2017},
  publisher={Github},
  journal={GitHub repository},
  howpublished={\url{https://github.com/matterport/Mask_RCNN}},
}